# Semi-Supervised Learning
 Mục tiêu
- Sử dụng cả dữ liệu có nhãn và không nhãn
- Dự đoán kết quả học tập

In [1]:
import pandas as pd
import numpy as np

from sklearn.semi_supervised import LabelPropagation
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Data loaded")


Data loaded


## Feature Scaling


In [3]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## Supervised Baseline (Random Forest)

In [4]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("=== Supervised Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


=== Supervised Random Forest ===
Accuracy: 0.6835443037974683
              precision    recall  f1-score   support

           0       0.54      0.27      0.36        26
           1       0.71      0.89      0.79        53

    accuracy                           0.68        79
   macro avg       0.63      0.58      0.57        79
weighted avg       0.65      0.68      0.65        79



## Simulate Unlabeled Data

In [5]:
y_semi = y_train.copy()

np.random.seed(42)

# 30% dữ liệu không có nhãn
mask = np.random.rand(len(y_semi)) < 0.3
y_semi[mask] = -1

print("Unlabeled samples:", sum(y_semi == -1))

Unlabeled samples: 99


## Train Model

In [6]:
model = LabelPropagation()

model.fit(X_train, y_semi)

,"kernel kernel: {'knn', 'rbf'} or callable, default='rbf'String identifier for kernel function to use or the kernel functionitself. Only 'rbf' and 'knn' strings are valid inputs. The functionpassed should take two inputs, each of shape (n_samples, n_features),and return a (n_samples, n_samples) shaped weight matrix.",'rbf'
,"gamma gamma: float, default=20Parameter for rbf kernel.",20
,"n_neighbors n_neighbors: int, default=7Parameter for knn kernel which need to be strictly positive.",7
,"max_iter max_iter: int, default=1000Change maximum number of iterations allowed.",1000
,"tol tol: float, default=1e-3Convergence tolerance: threshold to consider the system at steadystate.",0.001
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


## Predict on Test Set

In [7]:
y_pred_semi = model.predict(X_test)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/semi_supervised/_label_propagation.py:231: RuntimeWarning: invalid value encountered in divide
  probabilities /= normalizer


## Evaluation

In [8]:
accuracy_semi = accuracy_score(y_test, y_pred_semi)

print("=== Semi-Supervised (LabelPropagation) ===")
print("Accuracy:", accuracy_semi)
print(classification_report(y_test, y_pred_semi))

=== Semi-Supervised (LabelPropagation) ===
Accuracy: 0.5316455696202531
              precision    recall  f1-score   support

           0       0.35      0.50      0.41        26
           1       0.69      0.55      0.61        53

    accuracy                           0.53        79
   macro avg       0.52      0.52      0.51        79
weighted avg       0.58      0.53      0.55        79



## Confusion Matrix

In [9]:
cm = confusion_matrix(y_test, y_pred_semi)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[13 13]
 [24 29]]


## Model Comparison


In [10]:
print("\n=== Comparison ===")
print("Supervised RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Semi-supervised Accuracy:", accuracy_semi)
print("\n=> Supervised model vẫn tốt hơn, nhưng semi-supervised hữu ích khi thiếu dữ liệu nhãn.")


=== Comparison ===
Supervised RF Accuracy: 0.6835443037974683
Semi-supervised Accuracy: 0.5316455696202531

=> Supervised model vẫn tốt hơn, nhưng semi-supervised hữu ích khi thiếu dữ liệu nhãn.


## Insights

1. Semi-supervised learning cho phép tận dụng dữ liệu không có nhãn.

2. Với ~30% dữ liệu không nhãn, mô hình vẫn học được tương đối tốt.

3. So sánh cho thấy:
- Supervised thường cho kết quả tốt hơn khi có đủ dữ liệu nhãn
- Semi-supervised hữu ích khi dữ liệu nhãn bị hạn chế

4. Trong thực tế:
- Gán nhãn dữ liệu tốn chi phí và thời gian
- Semi-supervised giúp giảm phụ thuộc vào dữ liệu có nhãn

5. Ứng dụng:
- Dự đoán học sinh có nguy cơ rớt
- Xây dựng hệ thống cảnh báo sớm
